# 🎬 Movie Rental Data Warehouse — ETL Pipeline
### Data Warehousing Course — Final Assignment
---
This notebook implements a complete **ETL (Extract → Transform → Load)** pipeline for the Movie Rental Data Warehouse.

It reads raw operational data from the **Sakila OLTP MySQL database**, transforms it into a clean **Star Schema dimensional model**, validates every table using **Pandera data contracts**, and loads the result into a **SQLite Data Warehouse**.

---
**Warehouse Schema:**
- 2 Fact Tables: `fact_rental`, `fact_payment`
- 7 Dimension Tables: `dim_date`, `dim_customer`, `dim_film`, `dim_category`, `dim_actor`, `dim_store`, `dim_staff`
- 2 Bridge Tables: `bridge_film_category`, `bridge_film_actor`

**Pipeline Stages:**
1. Setup & Connections
2. Extract
3. Exploratory Data Analysis (EDA)
4. Transform → Validate → each table
5. Load
6. Verify
7. Analytical Queries (15 Business Questions)

## 1. Setup
Import libraries and establish connections to source OLTP database and target Data Warehouse.

In [ ]:
import pandas            as pd
import pandera.pandas    as pa
from sqlalchemy          import create_engine, text
from IPython.display     import display

# ── Connection to Sakila OLTP (MySQL) ──────────────────────────────────────
# Update credentials to match your local MySQL setup
MYSQL_USER     = "root"
MYSQL_PASSWORD = "12345"
MYSQL_HOST     = "127.0.0.1"
MYSQL_PORT     = "3306"
MYSQL_DB       = "sakila"

oltp_engine      = create_engine(
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DB}"
)

# ── Connection to Data Warehouse (SQLite — portable, no server needed) ──────
warehouse_engine = create_engine("sqlite:///movie_rental_dw.db")

print("✓ OLTP connection     : MySQL Sakila")
print("✓ Warehouse connection: SQLite movie_rental_dw.db")

## 2. Extract
Read all required source tables from the Sakila OLTP database into Pandas DataFrames.

**15 tables extracted** (film_text excluded — search-only table with no analytical value):

In [ ]:
# ── Extract all source tables ───────────────────────────────────────────────
rental_df         = pd.read_sql("SELECT * FROM rental"        , con=oltp_engine)
payment_df        = pd.read_sql("SELECT * FROM payment"       , con=oltp_engine)
customer_df       = pd.read_sql("SELECT * FROM customer"      , con=oltp_engine)
film_df           = pd.read_sql("SELECT * FROM film"          , con=oltp_engine)
inventory_df      = pd.read_sql("SELECT * FROM inventory"     , con=oltp_engine)
store_df          = pd.read_sql("SELECT * FROM store"         , con=oltp_engine)
staff_df          = pd.read_sql("SELECT * FROM staff"         , con=oltp_engine)
address_df        = pd.read_sql("SELECT * FROM address"       , con=oltp_engine)
city_df           = pd.read_sql("SELECT * FROM city"          , con=oltp_engine)
country_df        = pd.read_sql("SELECT * FROM country"       , con=oltp_engine)
category_df       = pd.read_sql("SELECT * FROM category"      , con=oltp_engine)
film_category_df  = pd.read_sql("SELECT * FROM film_category" , con=oltp_engine)
language_df       = pd.read_sql("SELECT * FROM language"      , con=oltp_engine)
actor_df          = pd.read_sql("SELECT * FROM actor"         , con=oltp_engine)
film_actor_df     = pd.read_sql("SELECT * FROM film_actor"    , con=oltp_engine)

# ── Extraction summary ───────────────────────────────────────────────────────
extracted = {
    "rental"        : rental_df,
    "payment"       : payment_df,
    "customer"      : customer_df,
    "film"          : film_df,
    "inventory"     : inventory_df,
    "store"         : store_df,
    "staff"         : staff_df,
    "address"       : address_df,
    "city"          : city_df,
    "country"       : country_df,
    "category"      : category_df,
    "film_category" : film_category_df,
    "language"      : language_df,
    "actor"         : actor_df,
    "film_actor"    : film_actor_df,
}

print(f"{'Table':<16} {'Rows':>6}  {'Columns':>7}")
print("─" * 34)
for name, df in extracted.items():
    print(f"  {name:<14} {df.shape[0]:>6,}  {df.shape[1]:>7}")
print("─" * 34)
print(f"  {'TOTAL':<14} {sum(d.shape[0] for d in extracted.values()):>6,}")

## 3. Exploratory Data Analysis (EDA)
Inspect structure, data types, nulls, and duplicates across all extracted tables before transforming.

In [ ]:
# ── Null and duplicate summary across all source tables ─────────────────────
print(f"{'Table':<16} {'Duplicates':>12}  {'Total Nulls':>12}")
print("─" * 44)
for name, df in extracted.items():
    dups  = df.duplicated().sum()
    nulls = df.isnull().sum().sum()
    flag  = " ⚠" if nulls > 0 else "  ✓"
    print(f"{flag} {name:<14} {dups:>12,}  {nulls:>12,}")

In [ ]:
# ── Inspect core fact source: rental ────────────────────────────────────────
print("rental table — first 5 rows:")
display(rental_df.head())
print()
print("rental table — column info:")
rental_df.info()

In [ ]:
# ── Null analysis on rental (return_date expected to have NULLs) ────────────
print("Null values per column in rental:")
display(rental_df.isnull().sum().rename("null_count").to_frame())

In [ ]:
# ── Inspect core fact source: payment ───────────────────────────────────────
print("payment table — first 5 rows:")
display(payment_df.head())

## 4. Transform
Clean, reshape, and enrich all source data into the dimensional model.
Each table is **built → validated** before proceeding to the next.

In [ ]:
# ── Reusable validation function with styled report ─────────────────────────
def validate(schema, dataframe, table_name=""):
    """Validate a DataFrame against a Pandera schema and display a styled report."""
    schema.validate(dataframe)

    report = []
    for col, col_schema in schema.columns.items():
        has_null  = dataframe[col].isnull().any()
        is_unique = dataframe[col].is_unique
        dtype     = str(dataframe[col].dtype)
        n_checks  = len(col_schema.checks)
        status    = "FAIL" if (has_null and not col_schema.nullable) else "PASS"
        report.append({
            "column"  : col,
            "dtype"   : dtype,
            "nulls"   : has_null,
            "unique"  : is_unique,
            "checks"  : n_checks,
            "status"  : status,
        })

    report_df = pd.DataFrame(report)
    title     = f"  Validation Report — {table_name}" if table_name else "  Validation Report"
    print(title)
    display(
        report_df.style.map(
            lambda v: "color: green; font-weight: bold" if v == "PASS"
                      else ("color: red; font-weight: bold" if v == "FAIL" else ""),
            subset=["status"]
        )
    )

### 4.1 Build `dim_date`
Collect all unique dates from rental, return, and payment columns.
Generate one row per calendar date with full time attributes.

**Special row:** `date_key = -1` → represents films **not yet returned** (NULL return_date in source).

In [ ]:
# ── Collect all dates from rental and payment data ──────────────────────────
rental_dates  = pd.to_datetime(rental_df["rental_date"] , errors="coerce")
return_dates  = pd.to_datetime(rental_df["return_date"] , errors="coerce")
payment_dates = pd.to_datetime(payment_df["payment_date"], errors="coerce")

all_dates      = pd.concat([rental_dates, return_dates, payment_dates]).dropna()
unique_dates   = pd.DatetimeIndex(
    pd.Series(all_dates.dt.date.unique()).sort_values().values
)

# ── Build dim_date ───────────────────────────────────────────────────────────
dim_date_df = pd.DataFrame({
    "date_key"    : unique_dates.strftime("%Y%m%d").astype(int),
    "full_date"   : unique_dates.date.astype(str),
    "day"         : unique_dates.day,
    "day_name"    : unique_dates.day_name(),
    "month"       : unique_dates.month,
    "month_name"  : unique_dates.month_name(),
    "quarter"     : unique_dates.quarter,
    "year"        : unique_dates.year,
    "is_weekend"  : unique_dates.dayofweek.isin([5, 6]).astype(int),
})

# ── Prepend special row for unreturned films ─────────────────────────────────
special_row = pd.DataFrame([{
    "date_key"  : -1,
    "full_date" : None,
    "day"       : None,
    "day_name"  : "Not Yet Returned",
    "month"     : None,
    "month_name": None,
    "quarter"   : None,
    "year"      : None,
    "is_weekend": None,
}])

dim_date_df = pd.concat([special_row, dim_date_df], ignore_index=True)

print(f"✓ dim_date built: {dim_date_df.shape[0]:,} rows  ({dim_date_df.shape[1]} columns)")
print(f"  Date range: {dim_date_df['full_date'].dropna().min()}  →  {dim_date_df['full_date'].dropna().max()}")
display(dim_date_df[dim_date_df["date_key"] >= 0].head())

#### Validate `dim_date`

In [ ]:
dim_date_schema = pa.DataFrameSchema({
    "date_key"  : pa.Column(int,    nullable=False, unique=True),
    "full_date" : pa.Column(object, nullable=True,  unique=False),
    "day"       : pa.Column(object, nullable=True),
    "day_name"  : pa.Column(str,    nullable=False),
    "month"     : pa.Column(object, nullable=True),
    "month_name": pa.Column(object, nullable=True),
    "quarter"   : pa.Column(object, nullable=True),
    "year"      : pa.Column(object, nullable=True),
    "is_weekend": pa.Column(object, nullable=True),
})

validate(dim_date_schema, dim_date_df, "dim_date")

### 4.2 Build `dim_customer`
Join `customer` → `address` → `city` → `country`.

**Transformations:**
- Flatten 3-level geographic hierarchy into single row
- Concatenate first + last name → `full_name` (title case)
- Fill NULL emails → `"unknown@unknown.com"`
- Convert `active` tinyint → `"Active"` / `"Inactive"`
- Add `customer_since` from `create_date`
- Generate surrogate key `customer_key`

In [ ]:
# ── Flatten address → city → country ────────────────────────────────────────
location_df = (
    address_df[["address_id", "address", "address2", "district", "postal_code", "phone", "city_id"]]
    .merge(city_df[["city_id", "city", "country_id"]]  , on="city_id"   , how="left")
    .merge(country_df[["country_id", "country"]]        , on="country_id", how="left")
)

# ── Build dim_customer ───────────────────────────────────────────────────────
dim_customer_df = customer_df.merge(location_df, on="address_id", how="left")

dim_customer_df["full_name"]       = (
    dim_customer_df["first_name"].str.strip().str.title() + " " +
    dim_customer_df["last_name"].str.strip().str.title()
)
dim_customer_df["email"]           = dim_customer_df["email"].fillna("unknown@unknown.com")
dim_customer_df["city"]            = dim_customer_df["city"].fillna("Unknown")
dim_customer_df["country"]         = dim_customer_df["country"].fillna("Unknown")
dim_customer_df["district"]        = dim_customer_df["district"].fillna("Unknown")
dim_customer_df["address"]         = dim_customer_df["address"].fillna("Unknown")
dim_customer_df["active_status"]   = dim_customer_df["active"].map({1: "Active", 0: "Inactive"})
dim_customer_df["customer_since"]  = pd.to_datetime(dim_customer_df["create_date"]).dt.date.astype(str)

dim_customer_df = (
    dim_customer_df[[
        "customer_id", "full_name", "email", "active_status",
        "address", "address2", "district", "city", "country",
        "postal_code", "phone", "customer_since"
    ]]
    .drop_duplicates(subset=["customer_id"])
    .reset_index(drop=True)
)
dim_customer_df.insert(0, "customer_key", dim_customer_df.index + 1)

print(f"✓ dim_customer built: {dim_customer_df.shape[0]:,} rows  ({dim_customer_df.shape[1]} columns)")
display(dim_customer_df.head())

#### Validate `dim_customer`

In [ ]:
dim_customer_schema = pa.DataFrameSchema({
    "customer_key"  : pa.Column(int, nullable=False, unique=True),
    "customer_id"   : pa.Column(int, nullable=False, unique=True),
    "full_name"     : pa.Column(str, nullable=False),
    "email"         : pa.Column(str, nullable=False),
    "active_status" : pa.Column(str, nullable=False,
                        checks=pa.Check.isin(["Active", "Inactive"])),
    "city"          : pa.Column(str, nullable=False),
    "country"       : pa.Column(str, nullable=False),
    "customer_since": pa.Column(str, nullable=False),
})

validate(dim_customer_schema, dim_customer_df, "dim_customer")

### 4.3 Build `dim_film`
Join `film` → `language` (twice: primary + original language).

**Transformations:**
- Embed language name (avoids separate dim_language table)
- Title-case film titles
- Keep `replacement_cost` for financial risk analysis
- Fill NULL ratings → `"Unrated"`
- Generate surrogate key `film_key`

In [ ]:
# ── Join language twice (primary + original) ────────────────────────────────
lang_primary  = language_df[["language_id", "name"]].rename(columns={"name": "language"})
lang_original = language_df[["language_id", "name"]].rename(
    columns={"language_id": "original_language_id", "name": "original_language"}
)

dim_film_df = (
    film_df
    .merge(lang_primary , on="language_id"         , how="left")
    .merge(lang_original, on="original_language_id", how="left")
)

dim_film_df["title"]             = dim_film_df["title"].str.strip().str.title()
dim_film_df["language"]          = dim_film_df["language"].fillna("Unknown")
dim_film_df["original_language"] = dim_film_df["original_language"].fillna(None)
dim_film_df["rating"]            = dim_film_df["rating"].fillna("Unrated")
dim_film_df["description"]       = dim_film_df["description"].fillna("")
dim_film_df["special_features"]  = dim_film_df["special_features"].fillna("")

dim_film_df = (
    dim_film_df[[
        "film_id", "title", "description", "release_year",
        "language", "original_language", "rental_rate",
        "rental_duration", "length", "replacement_cost",
        "rating", "special_features"
    ]]
    .drop_duplicates(subset=["film_id"])
    .reset_index(drop=True)
)
dim_film_df.insert(0, "film_key", dim_film_df.index + 1)

print(f"✓ dim_film built: {dim_film_df.shape[0]:,} rows  ({dim_film_df.shape[1]} columns)")
display(dim_film_df.head())

#### Validate `dim_film`

In [ ]:
dim_film_schema = pa.DataFrameSchema({
    "film_key"        : pa.Column(int,   nullable=False, unique=True),
    "film_id"         : pa.Column(int,   nullable=False, unique=True),
    "title"           : pa.Column(str,   nullable=False),
    "language"        : pa.Column(str,   nullable=False),
    "rating"          : pa.Column(str,   nullable=False),
    "rental_duration" : pa.Column(int,   nullable=False,
                            checks=pa.Check.greater_than(0)),
    "rental_rate"     : pa.Column(float, nullable=False,
                            checks=pa.Check.greater_than(0)),
    "replacement_cost": pa.Column(float, nullable=False,
                            checks=pa.Check.greater_than(0)),
})

validate(dim_film_schema, dim_film_df, "dim_film")

### 4.4 Build `dim_category`
Simple direct load from `category` table.
16 film categories in Sakila (Action, Comedy, Drama, etc.)

In [ ]:
dim_category_df = (
    category_df[["category_id", "name"]]
    .rename(columns={"name": "category_name"})
    .drop_duplicates(subset=["category_id"])
    .reset_index(drop=True)
)
dim_category_df.insert(0, "category_key", dim_category_df.index + 1)

print(f"✓ dim_category built: {dim_category_df.shape[0]:,} rows  ({dim_category_df.shape[1]} columns)")
display(dim_category_df)

#### Validate `dim_category`

In [ ]:
dim_category_schema = pa.DataFrameSchema({
    "category_key" : pa.Column(int, nullable=False, unique=True),
    "category_id"  : pa.Column(int, nullable=False, unique=True),
    "category_name": pa.Column(str, nullable=False, unique=True),
})

validate(dim_category_schema, dim_category_df, "dim_category")

### 4.5 Build `dim_actor`
Build from `actor` table. 200 actors in Sakila.

**Transformations:**
- Concatenate + proper-case first/last name → `full_name`
- Generate surrogate key `actor_key`

In [ ]:
dim_actor_df = actor_df[["actor_id", "first_name", "last_name"]].copy()

dim_actor_df["full_name"] = (
    dim_actor_df["first_name"].str.strip().str.title() + " " +
    dim_actor_df["last_name"].str.strip().str.title()
)

dim_actor_df = (
    dim_actor_df[["actor_id", "full_name"]]
    .drop_duplicates(subset=["actor_id"])
    .reset_index(drop=True)
)
dim_actor_df.insert(0, "actor_key", dim_actor_df.index + 1)

print(f"✓ dim_actor built: {dim_actor_df.shape[0]:,} rows  ({dim_actor_df.shape[1]} columns)")
display(dim_actor_df.head())

#### Validate `dim_actor`

In [ ]:
dim_actor_schema = pa.DataFrameSchema({
    "actor_key": pa.Column(int, nullable=False, unique=True),
    "actor_id" : pa.Column(int, nullable=False, unique=True),
    "full_name": pa.Column(str, nullable=False),
})

validate(dim_actor_schema, dim_actor_df, "dim_actor")

### 4.6 Build `dim_store`
Join `store` → `staff` (manager name) → `address` → `city` → `country`.

**Creative Addition:** `manager_full_name` embedded directly — store performance
reports immediately show which manager is responsible without extra joins.

In [ ]:
# ── Resolve manager name from staff ─────────────────────────────────────────
manager_df = staff_df[["staff_id", "first_name", "last_name"]].copy()
manager_df["manager_full_name"] = (
    manager_df["first_name"].str.strip().str.title() + " " +
    manager_df["last_name"].str.strip().str.title()
)
manager_df = manager_df.rename(columns={"staff_id": "manager_staff_id"})

# ── Join store → manager → address → city → country ─────────────────────────
dim_store_df = (
    store_df
    .merge(manager_df[["manager_staff_id", "manager_full_name"]], on="manager_staff_id", how="left")
    .merge(address_df[["address_id", "address", "address2", "district", "postal_code", "phone", "city_id"]], on="address_id", how="left")
    .merge(city_df[["city_id", "city", "country_id"]]           , on="city_id"   , how="left")
    .merge(country_df[["country_id", "country"]]                , on="country_id", how="left")
)

dim_store_df["city"]    = dim_store_df["city"].fillna("Unknown")
dim_store_df["country"] = dim_store_df["country"].fillna("Unknown")

dim_store_df = (
    dim_store_df[[
        "store_id", "manager_full_name",
        "address", "address2", "district",
        "city", "country", "postal_code", "phone"
    ]]
    .drop_duplicates(subset=["store_id"])
    .reset_index(drop=True)
)
dim_store_df.insert(0, "store_key", dim_store_df.index + 1)

print(f"✓ dim_store built: {dim_store_df.shape[0]:,} rows  ({dim_store_df.shape[1]} columns)")
display(dim_store_df)

#### Validate `dim_store`

In [ ]:
dim_store_schema = pa.DataFrameSchema({
    "store_key"         : pa.Column(int, nullable=False, unique=True),
    "store_id"          : pa.Column(int, nullable=False, unique=True),
    "manager_full_name" : pa.Column(str, nullable=False),
    "city"              : pa.Column(str, nullable=False),
    "country"           : pa.Column(str, nullable=False),
})

validate(dim_store_schema, dim_store_df, "dim_store")

### 4.7 Build `dim_staff`
Build from `staff` table.

**Transformations:**
- Concatenate + proper-case name
- Convert active flag → `"Active"` / `"Inactive"`
- Fill NULL emails → `"unknown@unknown.com"`

In [ ]:
dim_staff_df = staff_df[["staff_id", "first_name", "last_name", "email", "store_id", "active"]].copy()

dim_staff_df["full_name"]     = (
    dim_staff_df["first_name"].str.strip().str.title() + " " +
    dim_staff_df["last_name"].str.strip().str.title()
)
dim_staff_df["email"]         = dim_staff_df["email"].fillna("unknown@unknown.com")
dim_staff_df["active_status"] = dim_staff_df["active"].map({1: "Active", 0: "Inactive"})

dim_staff_df = (
    dim_staff_df[["staff_id", "full_name", "email", "active_status", "store_id"]]
    .drop_duplicates(subset=["staff_id"])
    .reset_index(drop=True)
)
dim_staff_df.insert(0, "staff_key", dim_staff_df.index + 1)

print(f"✓ dim_staff built: {dim_staff_df.shape[0]:,} rows  ({dim_staff_df.shape[1]} columns)")
display(dim_staff_df)

#### Validate `dim_staff`

In [ ]:
dim_staff_schema = pa.DataFrameSchema({
    "staff_key"    : pa.Column(int, nullable=False, unique=True),
    "staff_id"     : pa.Column(int, nullable=False, unique=True),
    "full_name"    : pa.Column(str, nullable=False),
    "email"        : pa.Column(str, nullable=False),
    "active_status": pa.Column(str, nullable=False,
                        checks=pa.Check.isin(["Active", "Inactive"])),
    "store_id"     : pa.Column(int, nullable=False),
})

validate(dim_staff_schema, dim_staff_df, "dim_staff")

### 4.8 Build Bridge Tables
Bridge tables resolve **many-to-many** relationships between `dim_film` and
`dim_category` / `dim_actor`. They replace OLTP natural keys with DW surrogate keys.

In [ ]:
# ── bridge_film_category ────────────────────────────────────────────────────
bridge_film_category_df = (
    film_category_df[["film_id", "category_id"]]
    .merge(dim_film_df[["film_key", "film_id"]]         , on="film_id"    , how="inner")
    .merge(dim_category_df[["category_key", "category_id"]], on="category_id", how="inner")
    [["film_key", "category_key"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

# ── bridge_film_actor ────────────────────────────────────────────────────────
bridge_film_actor_df = (
    film_actor_df[["film_id", "actor_id"]]
    .merge(dim_film_df[["film_key", "film_id"]]    , on="film_id" , how="inner")
    .merge(dim_actor_df[["actor_key", "actor_id"]] , on="actor_id", how="inner")
    [["film_key", "actor_key"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"✓ bridge_film_category built: {bridge_film_category_df.shape[0]:,} rows")
print(f"✓ bridge_film_actor built   : {bridge_film_actor_df.shape[0]:,} rows")
display(bridge_film_category_df.head())
display(bridge_film_actor_df.head())

### 4.9 Build `fact_rental`
**Grain:** One row per rental transaction.

**Key Transformations:**
- `rental` → `inventory` to resolve `film_id` and `store_id`
- Map all OLTP IDs → DW surrogate keys
- Derive `actual_duration_days` = return_date − rental_date
- Derive `expected_duration_days` from `film.rental_duration`
- Derive `days_overdue` = max(0, actual − expected)
- Derive `is_late_return` flag
- Handle NULL `return_date` → `is_returned = 0`, `date_key_return = -1`

In [ ]:
# ── Build surrogate key lookup maps ─────────────────────────────────────────
date_lookup     = dim_date_df.set_index("full_date")["date_key"]
customer_lookup = dim_customer_df.set_index("customer_id")["customer_key"]
film_lookup     = dim_film_df.set_index("film_id")["film_key"]
store_lookup    = dim_store_df.set_index("store_id")["store_key"]
staff_lookup    = dim_staff_df.set_index("staff_id")["staff_key"]
duration_lookup = dim_film_df.set_index("film_key")["rental_duration"]

# ── Merge rental with inventory to get film_id + store_id ───────────────────
rental_enriched = rental_df.merge(
    inventory_df[["inventory_id", "film_id", "store_id"]],
    on="inventory_id", how="left"
)

# ── Parse dates ──────────────────────────────────────────────────────────────
rental_enriched["rental_date_parsed"]  = pd.to_datetime(rental_enriched["rental_date"] , errors="coerce")
rental_enriched["return_date_parsed"]  = pd.to_datetime(rental_enriched["return_date"] , errors="coerce")

# ── Map surrogate keys ───────────────────────────────────────────────────────
rental_enriched["date_key_rental"]  = (rental_enriched["rental_date_parsed"]
                                        .dt.date.astype(str).map(date_lookup))
rental_enriched["date_key_return"]  = (rental_enriched["return_date_parsed"]
                                        .dt.date.astype(str).map(date_lookup)
                                        .fillna(-1).astype(int))
rental_enriched["customer_key"]     = rental_enriched["customer_id"].map(customer_lookup)
rental_enriched["film_key"]         = rental_enriched["film_id"].map(film_lookup)
rental_enriched["store_key"]        = rental_enriched["store_id"].map(store_lookup)
rental_enriched["staff_key"]        = rental_enriched["staff_id"].map(staff_lookup)

# ── Derive measures ──────────────────────────────────────────────────────────
rental_enriched["actual_duration_days"]   = (
    (rental_enriched["return_date_parsed"] - rental_enriched["rental_date_parsed"]).dt.days
)
rental_enriched["expected_duration_days"] = (
    rental_enriched["film_key"].map(duration_lookup)
)
rental_enriched["days_overdue"]           = (
    (rental_enriched["actual_duration_days"] - rental_enriched["expected_duration_days"])
    .clip(lower=0)
)
rental_enriched["is_late_return"]         = (
    (rental_enriched["actual_duration_days"] > rental_enriched["expected_duration_days"])
    .astype("Int64")
)
rental_enriched["is_returned"]            = rental_enriched["return_date"].notna().astype(int)
rental_enriched["rental_count"]           = 1

# ── Select final columns ─────────────────────────────────────────────────────
fact_rental_df = (
    rental_enriched[[
        "rental_id", "date_key_rental", "date_key_return",
        "customer_key", "film_key", "store_key", "staff_key",
        "actual_duration_days", "expected_duration_days",
        "days_overdue", "is_late_return", "is_returned", "rental_count"
    ]]
    .drop_duplicates(subset=["rental_id"])
    .dropna(subset=["customer_key", "film_key", "store_key", "staff_key"])
    .reset_index(drop=True)
)

print(f"✓ fact_rental built: {fact_rental_df.shape[0]:,} rows  ({fact_rental_df.shape[1]} columns)")
print(f"  Returned    : {fact_rental_df['is_returned'].sum():,}")
print(f"  Not returned: {(fact_rental_df['is_returned'] == 0).sum():,}")
print(f"  Late returns: {fact_rental_df['is_late_return'].sum():,}")
display(fact_rental_df.head())

#### Validate `fact_rental`

In [ ]:
fact_rental_schema = pa.DataFrameSchema({
    "rental_id"             : pa.Column(int,   nullable=False, unique=True),
    "date_key_rental"       : pa.Column(float, nullable=False),
    "date_key_return"       : pa.Column(int,   nullable=False),
    "customer_key"          : pa.Column(float, nullable=False),
    "film_key"              : pa.Column(float, nullable=False),
    "store_key"             : pa.Column(float, nullable=False),
    "staff_key"             : pa.Column(float, nullable=False),
    "actual_duration_days"  : pa.Column(float, nullable=True),
    "expected_duration_days": pa.Column(float, nullable=True),
    "days_overdue"          : pa.Column(float, nullable=True),
    "is_late_return"        : pa.Column(object,nullable=True),
    "is_returned"           : pa.Column(int,   nullable=False,
                                checks=pa.Check.isin([0, 1])),
    "rental_count"          : pa.Column(int,   nullable=False,
                                checks=pa.Check.equal_to(1)),
})

validate(fact_rental_schema, fact_rental_df, "fact_rental")

### 4.10 Build `fact_payment`
**Grain:** One row per payment transaction.

**Key note:** `store_key` resolved via `payment → staff → store`
since the OLTP payment table has no direct store_id column.

In [ ]:
# ── Resolve store via payment → rental → inventory → store ──────────────────
payment_enriched = (
    payment_df
    .merge(rental_df[["rental_id", "inventory_id"]]              , on="rental_id"   , how="left")
    .merge(inventory_df[["inventory_id", "film_id", "store_id"]] , on="inventory_id", how="left")
    .merge(staff_df[["staff_id", "store_id"]].rename(
        columns={"store_id": "staff_store_id"})                   , on="staff_id"    , how="left")
)

# ── Use store from inventory if available, else fall back to staff store ──────
payment_enriched["resolved_store_id"] = (
    payment_enriched["store_id"].fillna(payment_enriched["staff_store_id"])
)

# ── Parse payment date ───────────────────────────────────────────────────────
payment_enriched["payment_date_parsed"] = pd.to_datetime(
    payment_enriched["payment_date"], errors="coerce"
)

# ── Map surrogate keys ───────────────────────────────────────────────────────
payment_enriched["date_key"]      = (payment_enriched["payment_date_parsed"]
                                      .dt.date.astype(str).map(date_lookup))
payment_enriched["customer_key"]  = payment_enriched["customer_id"].map(customer_lookup)
payment_enriched["staff_key"]     = payment_enriched["staff_id"].map(staff_lookup)
payment_enriched["store_key"]     = payment_enriched["resolved_store_id"].map(store_lookup)
payment_enriched["film_key"]      = payment_enriched["film_id"].map(film_lookup)
payment_enriched["payment_count"] = 1

# ── Select final columns ─────────────────────────────────────────────────────
fact_payment_df = (
    payment_enriched[[
        "payment_id", "date_key", "customer_key",
        "staff_key", "store_key", "film_key",
        "rental_id", "amount", "payment_count"
    ]]
    .rename(columns={"amount": "payment_amount"})
    .drop_duplicates(subset=["payment_id"])
    .dropna(subset=["customer_key"])
    .reset_index(drop=True)
)

print(f"✓ fact_payment built: {fact_payment_df.shape[0]:,} rows  ({fact_payment_df.shape[1]} columns)")
print(f"  Total revenue: ${fact_payment_df['payment_amount'].sum():,.2f}")
display(fact_payment_df.head())

#### Validate `fact_payment`

In [ ]:
fact_payment_schema = pa.DataFrameSchema({
    "payment_id"    : pa.Column(int,   nullable=False, unique=True),
    "date_key"      : pa.Column(float, nullable=True),
    "customer_key"  : pa.Column(float, nullable=False),
    "staff_key"     : pa.Column(float, nullable=False),
    "store_key"     : pa.Column(float, nullable=True),
    "film_key"      : pa.Column(float, nullable=True),
    "rental_id"     : pa.Column(object,nullable=True),
    "payment_amount": pa.Column(float, nullable=False,
                        checks=pa.Check.greater_than_or_equal_to(0)),
    "payment_count" : pa.Column(int,   nullable=False,
                        checks=pa.Check.equal_to(1)),
})

validate(fact_payment_schema, fact_payment_df, "fact_payment")

## 5. Load
Load all tables into the Data Warehouse in the correct order.

**Critical rule:** Dimension tables must load **before** fact tables.
Bridge tables must load after their parent dimensions.

In [ ]:
# ── Reusable load function ───────────────────────────────────────────────────
def load_table(df, table_name, engine):
    df.to_sql(name=table_name, con=engine, if_exists="replace", index=False)
    print(f"  ✓ {table_name:<25} {df.shape[0]:>7,} rows loaded")

print("Loading Data Warehouse tables...")
print("─" * 45)

# Step 1-2: Date and Customer (no dimension dependencies)
load_table(dim_date_df            , "dim_date"            , warehouse_engine)
load_table(dim_customer_df        , "dim_customer"        , warehouse_engine)

# Step 3-5: Film and related dimensions
load_table(dim_film_df            , "dim_film"            , warehouse_engine)
load_table(dim_category_df        , "dim_category"        , warehouse_engine)
load_table(dim_actor_df           , "dim_actor"           , warehouse_engine)

# Step 6-7: Bridge tables (depend on dim_film, dim_category, dim_actor)
load_table(bridge_film_category_df, "bridge_film_category", warehouse_engine)
load_table(bridge_film_actor_df   , "bridge_film_actor"   , warehouse_engine)

# Step 8-9: Store and Staff
load_table(dim_store_df           , "dim_store"           , warehouse_engine)
load_table(dim_staff_df           , "dim_staff"           , warehouse_engine)

# Step 10-11: Fact tables (depend on ALL dimensions above)
load_table(fact_rental_df         , "fact_rental"         , warehouse_engine)
load_table(fact_payment_df        , "fact_payment"        , warehouse_engine)

print("─" * 45)
print("✓ All tables loaded successfully.")

## 6. Verify
Confirm all tables loaded correctly. Cross-check record counts and revenue totals against OLTP source.

In [ ]:
# ── Warehouse row count verification ────────────────────────────────────────
all_tables = [
    "dim_date", "dim_customer", "dim_film", "dim_category",
    "dim_actor", "bridge_film_category", "bridge_film_actor",
    "dim_store", "dim_staff", "fact_rental", "fact_payment"
]

print(f"{'Table':<26} {'DW Rows':>10}")
print("─" * 40)
with warehouse_engine.connect() as conn:
    for t in all_tables:
        count = conn.execute(text(f"SELECT COUNT(*) FROM {t}")).fetchone()[0]
        print(f"  {t:<24} {count:>10,}")

In [ ]:
# ── Reconciliation: DW vs OLTP totals ───────────────────────────────────────
oltp_rentals  = len(rental_df)
dw_rentals    = len(fact_rental_df)

oltp_payments = len(payment_df)
dw_payments   = len(fact_payment_df)

oltp_revenue  = payment_df["amount"].sum()
dw_revenue    = fact_payment_df["payment_amount"].sum()

print("Reconciliation Report")
print("─" * 52)
print(f"  {'Metric':<28} {'OLTP':>10}  {'DW':>10}")
print("─" * 52)
print(f"  {'Total Rentals':<28} {oltp_rentals:>10,}  {dw_rentals:>10,}  {'✓' if oltp_rentals == dw_rentals else '✗'}")
print(f"  {'Total Payments':<28} {oltp_payments:>10,}  {dw_payments:>10,}  {'✓' if oltp_payments == dw_payments else '✗'}")
print(f"  {'Total Revenue ($)':<28} {oltp_revenue:>10,.2f}  {dw_revenue:>10,.2f}  {'✓' if abs(oltp_revenue - dw_revenue) < 0.01 else '✗'}")
print("─" * 52)

## 7. Analytical Queries — 15 Business Questions
All queries run against the Data Warehouse using SQL via SQLAlchemy.

### Q1: Which films are rented most frequently?

In [ ]:
pd.read_sql("""
SELECT   f.title, f.rating, f.language,
         SUM(r.rental_count) AS total_rentals
FROM     fact_rental r
JOIN     dim_film f ON r.film_key = f.film_key
GROUP BY f.film_key, f.title, f.rating, f.language
ORDER BY total_rentals DESC
LIMIT    10
""", con=warehouse_engine)

### Q2: Which films generate the highest revenue?

In [ ]:
pd.read_sql("""
SELECT   f.title, f.rating,
         ROUND(SUM(p.payment_amount), 2) AS total_revenue,
         COUNT(p.payment_id)             AS total_payments
FROM     fact_payment p
JOIN     dim_film f ON p.film_key = f.film_key
GROUP BY f.film_key, f.title, f.rating
ORDER BY total_revenue DESC
LIMIT    10
""", con=warehouse_engine)

### Q3: Which film categories are most popular?

In [ ]:
pd.read_sql("""
SELECT   c.category_name,
         SUM(r.rental_count)            AS total_rentals,
         COUNT(DISTINCT r.customer_key) AS unique_customers
FROM     fact_rental r
JOIN     bridge_film_category bfc ON r.film_key      = bfc.film_key
JOIN     dim_category         c   ON bfc.category_key = c.category_key
GROUP BY c.category_key, c.category_name
ORDER BY total_rentals DESC
""", con=warehouse_engine)

### Q4: Which stores generate the highest number of rentals?

In [ ]:
pd.read_sql("""
SELECT   s.store_id, s.city, s.country, s.manager_full_name,
         SUM(r.rental_count) AS total_rentals
FROM     fact_rental r
JOIN     dim_store s ON r.store_key = s.store_key
GROUP BY s.store_key, s.store_id, s.city, s.country, s.manager_full_name
ORDER BY total_rentals DESC
""", con=warehouse_engine)

### Q5: Which stores generate the highest revenue?

In [ ]:
pd.read_sql("""
SELECT   s.store_id, s.city, s.country, s.manager_full_name,
         ROUND(SUM(p.payment_amount), 2) AS total_revenue
FROM     fact_payment p
JOIN     dim_store s ON p.store_key = s.store_key
GROUP BY s.store_key, s.store_id, s.city, s.country, s.manager_full_name
ORDER BY total_revenue DESC
""", con=warehouse_engine)

### Q6: Which customers rent the most films?

In [ ]:
pd.read_sql("""
SELECT   c.full_name, c.city, c.country, c.active_status,
         SUM(r.rental_count) AS total_rentals
FROM     fact_rental r
JOIN     dim_customer c ON r.customer_key = c.customer_key
GROUP BY c.customer_key, c.full_name, c.city, c.country, c.active_status
ORDER BY total_rentals DESC
LIMIT    10
""", con=warehouse_engine)

### Q7: Which customers generate the highest revenue?

In [ ]:
pd.read_sql("""
SELECT   c.full_name, c.city, c.country,
         ROUND(SUM(p.payment_amount), 2) AS total_revenue,
         COUNT(p.payment_id)             AS total_payments
FROM     fact_payment p
JOIN     dim_customer c ON p.customer_key = c.customer_key
GROUP BY c.customer_key, c.full_name, c.city, c.country
ORDER BY total_revenue DESC
LIMIT    10
""", con=warehouse_engine)

### Q8: How does rental activity change over time?

In [ ]:
pd.read_sql("""
SELECT   d.year, d.month, d.month_name,
         SUM(r.rental_count) AS total_rentals
FROM     fact_rental r
JOIN     dim_date d ON r.date_key_rental = d.date_key
GROUP BY d.year, d.month, d.month_name
ORDER BY d.year, d.month
""", con=warehouse_engine)

### Q9: How does revenue change by month, quarter, and year?

In [ ]:
pd.read_sql("""
SELECT   d.year, d.quarter, d.month, d.month_name,
         ROUND(SUM(p.payment_amount), 2) AS total_revenue
FROM     fact_payment p
JOIN     dim_date d ON p.date_key = d.date_key
GROUP BY d.year, d.quarter, d.month, d.month_name
ORDER BY d.year, d.month
""", con=warehouse_engine)

### Q10: Which staff members process the most rentals and revenue?

In [ ]:
pd.read_sql("""
SELECT   st.full_name, st.active_status,
         SUM(r.rental_count)            AS total_rentals,
         ROUND(SUM(p.payment_amount),2) AS total_revenue
FROM     dim_staff st
LEFT JOIN fact_rental  r ON st.staff_key = r.staff_key
LEFT JOIN fact_payment p ON st.staff_key = p.staff_key
GROUP BY st.staff_key, st.full_name, st.active_status
ORDER BY total_rentals DESC
""", con=warehouse_engine)

### Q11: Which cities and countries have the most active customers?

In [ ]:
pd.read_sql("""
SELECT   c.country, c.city,
         COUNT(DISTINCT r.customer_key) AS unique_customers,
         SUM(r.rental_count)            AS total_rentals
FROM     fact_rental r
JOIN     dim_customer c ON r.customer_key = c.customer_key
GROUP BY c.country, c.city
ORDER BY total_rentals DESC
LIMIT    15
""", con=warehouse_engine)

### Q12: What is the average rental duration by category?

In [ ]:
pd.read_sql("""
SELECT   c.category_name,
         ROUND(AVG(r.actual_duration_days), 2) AS avg_actual_days,
         MAX(f.rental_duration)                AS allowed_days,
         COUNT(r.rental_id)                    AS total_rentals
FROM     fact_rental r
JOIN     dim_film           f   ON r.film_key      = f.film_key
JOIN     bridge_film_category bfc ON r.film_key    = bfc.film_key
JOIN     dim_category       c   ON bfc.category_key = c.category_key
WHERE    r.is_returned = 1
GROUP BY c.category_key, c.category_name
ORDER BY avg_actual_days DESC
""", con=warehouse_engine)

### Q13: Which films are returned late most often?

In [ ]:
pd.read_sql("""
SELECT   f.title, f.rental_duration AS allowed_days,
         COUNT(r.rental_id)                              AS total_rentals,
         SUM(r.is_late_return)                           AS late_returns,
         ROUND(SUM(r.is_late_return)*100.0/COUNT(r.rental_id), 1) AS late_pct,
         ROUND(AVG(r.days_overdue), 1)                   AS avg_days_overdue
FROM     fact_rental r
JOIN     dim_film f ON r.film_key = f.film_key
WHERE    r.is_returned = 1
GROUP BY f.film_key, f.title, f.rental_duration
HAVING   late_returns > 0
ORDER BY late_pct DESC
LIMIT    10
""", con=warehouse_engine)

### Q14: Which actors appear in the most frequently rented films?

In [ ]:
pd.read_sql("""
SELECT   a.full_name AS actor_name,
         COUNT(DISTINCT bfa.film_key) AS films_count,
         SUM(r.rental_count)          AS total_rentals
FROM     dim_actor a
JOIN     bridge_film_actor bfa ON a.actor_key  = bfa.actor_key
JOIN     fact_rental       r   ON bfa.film_key = r.film_key
GROUP BY a.actor_key, a.full_name
ORDER BY total_rentals DESC
LIMIT    10
""", con=warehouse_engine)

### Q15: How does store performance differ by location? (Full KPI view)

In [ ]:
pd.read_sql("""
SELECT   s.store_id, s.city, s.country, s.manager_full_name,
         SUM(r.rental_count)                                  AS total_rentals,
         ROUND(SUM(p.payment_amount), 2)                      AS total_revenue,
         COUNT(DISTINCT r.customer_key)                        AS unique_customers,
         ROUND(SUM(p.payment_amount)/MAX(SUM(r.rental_count)),2) AS avg_rev_per_rental,
         SUM(r.is_late_return)                                 AS total_late_returns
FROM     fact_rental r
JOIN     dim_store    s  ON r.store_key = s.store_key
LEFT JOIN fact_payment p ON r.store_key = p.store_key
GROUP BY s.store_key, s.store_id, s.city, s.country, s.manager_full_name
ORDER BY total_revenue DESC
""", con=warehouse_engine)

### 🎁 Bonus: Unreturned Films — Financial Risk Analysis
Which films have not been returned? What is the total replacement value at risk?

In [ ]:
pd.read_sql("""
SELECT   f.title, f.replacement_cost,
         c.full_name  AS customer_name,
         c.email, c.phone, c.city,
         d.full_date  AS rental_date
FROM     fact_rental r
JOIN     dim_film     f ON r.film_key       = f.film_key
JOIN     dim_customer c ON r.customer_key   = c.customer_key
JOIN     dim_date     d ON r.date_key_rental = d.date_key
WHERE    r.is_returned = 0
ORDER BY f.replacement_cost DESC
""", con=warehouse_engine)

In [ ]:
# ── Total financial exposure from unreturned films ───────────────────────────
unreturned = pd.read_sql("""
    SELECT SUM(f.replacement_cost) AS total_replacement_value,
           COUNT(*)                AS unreturned_count
    FROM   fact_rental r
    JOIN   dim_film f ON r.film_key = f.film_key
    WHERE  r.is_returned = 0
""", con=warehouse_engine)

print(f"Unreturned films      : {int(unreturned['unreturned_count'][0]):,}")
print(f"Total value at risk   : ${float(unreturned['total_replacement_value'][0]):,.2f}")